# V7P3R Heuristics Consolidation (v1-v8)

**Complete Documentation of ALL Heuristic Concepts Across 8 Generations**

This notebook consolidates every heuristic, feature, evaluation metric, and personality trait ever implemented in V7P3R's static evaluation engines and neural network architectures.

---

## Overview: Heuristic Categories

The V7P3R heuristic vision is organized into **5 core categories**:

### 1. **Stage 1 Fast Features (19 dims)** — Piece-Centric Core
- Immediate, computed in O(1) or O(n) time
- Foundational for both static eval and neural input

### 2. **Heuristic Features (24 dims)** — Positional Quality Metrics
- Pawn structure analysis (passed, doubled, isolated, connected)
- Piece activity (rooks on 7th rank, bishop pairs, development)
- King safety metrics (pawn shield, exposure, escape squares)

### 3. **Complexity Features (8 dims)** — Game Difficulty & Phase
- **Forest Darkness**: V7P3R's signature complexity metric
- Piece tension, move diversity, center control
- Game phase detection (opening → midgame → endgame)

### 4. **Temporal Features (4 dims)** — Time & Situation Context
- Move urgency, time pressure indicators
- Halfmove clock (proximity to 50-move rule)
- Previous inference time feedback

### 5. **Static Evaluator Heuristics** — V7P3R Engine-Specific
- King safety evaluator (pawn shelter, escape squares, attack zones)
- Pawn structure evaluator (advanced patterns, chains, storms)
- Personality reward system (Tal-style complexity seeking)

---

## Part 1: Stage 1 Fast Features (19 dims)

**Source**: v6.0 onwards (lightweight, training input)

These are the **foundational features** computed first. They're blazingly fast and capture the immediate material and positional state.

### 1.1 Piece Counts (12 dims)
- White: Pawns, Knights, Bishops, Rooks, Queens, Kings (6 counts)
- Black: Same 6 counts
- Why: Piece material is absolute truth for evaluation

### 1.2 Material Balance (1 dim)
```
material_balance = white_material - black_material
where: Pawn=1, Knight=3, Bishop=3, Rook=5, Queen=9
```
- **Why**: Direct evaluation proxy (material advantage = winning advantage)
- **Use**: Move ordering, alpha-beta pruning heuristics

### 1.3 Side to Move (1 dim)
```
side_to_move = 1 if white_to_move else -1
```
- **Why**: Critical context for perspective-aware evaluation
- **Use**: Flip perspective for symmetric positions

### 1.4 Castling Rights (2 dims)
- White kingside castling rights: 1 if available, 0 otherwise
- White queenside castling rights: 1 if available, 0 otherwise
- **Why**: King safety indicator; castling availability changes evaluation
- **Missing**: Black castling (added to feature extraction in later versions)

### 1.5 In Check (1 dim)
```
in_check = 1 if current_side_in_check else 0
```
- **Why**: Immediate threat assessment
- **Use**: Reduce search depth in check (less pruning), adjust eval

### 1.6 Mobility (2 dims)
- `current_mobility`: Legal move count for side-to-move
- `opponent_mobility`: Legal move count for opponent
- **Why**: Piece activity proxy; more moves = more options = better position
- **Scale**: Typical range 20-50 moves mid-game

---

## Part 2: Heuristic Features (24 dims)

**Source**: v7.0 comprehensive features, v12.4 static eval

These are **positional quality metrics** that distinguish good positions from bad ones. They capture the structural chess principles humans learn.

### 2.1 Pawn Structure Features (8 dims)

#### Bishop Pair (2 dims)
```python
white_bishop_pair = 1 if white_has_both_bishops else 0
black_bishop_pair = 1 if black_has_both_bishops else 0
```
- **Chess Principle**: Two bishops > two knights or bishop+knight in open positions
- **Strength**: Especially valuable in open positions with many pawns
- **Weakness**: Useless in blocked/closed positions
- **V7P3R Personality**: Relatively standard, matches Stockfish approach

#### Passed Pawns (2 dims)
```python
white_passed_pawns = count(pawns_not_opposed_on_file_or_adjacent_files)
black_passed_pawns = count(pawns_not_opposed_on_file_or_adjacent_files)
```
- **Algorithm**: For each pawn, check if opponent has pawn on current/adjacent files ahead
- **Chess Principle**: Passed pawns are powerful; further advanced = stronger
- **Endgame Critical**: Passed pawn evaluation (by rank) in v12.4:
  - Rank 2-3: +20 cp
  - Rank 4: +30 cp
  - Rank 5: +50 cp
  - Rank 6: +80 cp
  - Rank 7: +120 cp
  - Rank 8: +250 cp (about to promote)
- **Advanced Bonus**: +30 cp if passed pawn on 6th rank or higher
- **Connected Passed**: +20 cp bonus if two passed pawns supporting each other

#### Doubled Pawns (2 dims)
```python
white_doubled = count(files_with_2+_pawns)
black_doubled = count(files_with_2+_pawns)
```
- **Penalty**: -25 cp per doubled pawn
- **Worse On**: Open files (-10 cp additional penalty)
- **Why Bad**: Reduced pawn coverage, less mobile
- **Exception**: Sometimes doubled pawns support piece activity

#### Isolated Pawns (2 dims)
```python
white_isolated = count(pawns_with_no_pawns_on_adjacent_files)
black_isolated = count(pawns_with_no_pawns_on_adjacent_files)
```
- **Penalty**: -15 cp per isolated pawn
- **Worse On**: Semi-open files where opponent can attack (-8 cp additional)
- **Why Bad**: Cannot be defended by other pawns; target for attack
- **Exception**: Isolated pawns can be strong if piece-supported

### 2.2 Piece Activity Features (8 dims)

#### King Pawn Shield (2 dims)
```python
def evaluate_king_pawn_shield(board, color):
    """Count pawns within 2 squares of king"""
    king_pos = board.king(color)
    shield_score = count(pawns_in_3x3_around_king)
    return shield_score  # 0-3 typically
```
- **Bonuses** (v12.4):
  - 0 shield pawns: 0 cp
  - 1 shield pawn: +5 cp
  - 2 shield pawns: +10 cp
  - 3 shield pawns: +15 cp
  - 4 shield pawns: +20 cp
- **Missing Shelter**: -10 cp per missing file
- **Pawn Storms Against**: -15 cp per enemy pawn storm
- **Why**: Pawn shield protects king from tactics

#### Active Rooks (2 dims)
```python
def count_active_rooks(board, color):
    """Rooks on 7th/8th rank OR on open/semi-open files"""
    active = 0
    for rook_square in board.rooks(color):
        if rook_on_7th_rank or (rook_on_open_file and not_blocked_by_pawns):
            active += 1
    return active
```
- **Bonus**: +4 cp per active rook
- **Why**: Rooks are strongest on advanced ranks or open files
- **Principle**: "A rook on the 7th is worth its weight in gold"

#### Development Score (2 dims)
```python
def count_development(board, color):
    """Count knights, bishops, queens not on starting rank"""
    developed = 0
    for piece_type in [KNIGHT, BISHOP, QUEEN]:
        developed += count(pieces_not_on_starting_rank)
    return developed
```
- **Bonus**: +1 cp per developed piece (typically 0-13 range)
- **Opening Importance**: Early development is critical for initiative
- **Why**: More pieces in play = more attacking/defending power

#### Piece Mobility (Normalized) (2 dims)
```python
def evaluate_mobility_normalized(board):
    """Calculate attacked squares / 64 for both sides"""
    white_attacked = count(squares_attacked_by_white)
    black_attacked = count(squares_attacked_by_black)
    return white_attacked/64, black_attacked/64
```
- **Range**: 0.0 to 1.0 per side
- **Mid-game Typical**: 0.4-0.6 (25-40 squares controlled)
- **Bonus**: Higher mobility = better position control

### 2.3 Relative Advantages (8 dims - from Side-to-Move Perspective)

These are **perspective-aware** versions of the features above, flipped from the mover's viewpoint:

```python
bishop_pair_advantage = (white_bishop_pair - black_bishop_pair) * perspective
passed_pawns_advantage = (white_passed - black_passed) * perspective
doubled_pawns_disadvantage = (black_doubled - white_doubled) * perspective  # More doubled = bad
isolated_pawns_disadvantage = (black_isolated - white_isolated) * perspective
king_safety_advantage = (white_shield - black_shield) * perspective
active_rooks_advantage = (white_rooks - black_rooks) * perspective
development_advantage = (white_dev - black_dev) * perspective
mobility_advantage = (white_mob - black_mob) * perspective
```

- **Why Relative**: Model sees position from moving side's perspective
- **Perspective**: +1 if white to move, -1 if black to move
- **Result**: All features become "is my side winning this aspect?"

---

## Part 3: Complexity Features (8 dims)

**Source**: v7.0 comprehensive features, v7.1 generational training

These capture the **psychological difficulty** of a position—how chaotic, tactical, or unclear it is.

### 3.1 Legal Moves Count (1 dim)
```python
legal_moves = count(board.legal_moves)  # Typical range: 20-50
```
- **Typical Values**:
  - Constrained position (restricted king): 10-20
  - Normal mid-game: 30-40
  - Open mid-game: 40-50
  - Restricted (end-game): 5-20
- **Why**: More moves = more options = harder to play optimally
- **Normalized**: scaled relative to 60 (max in typical position)

### 3.2 Captures Available (1 dim)
```python
captures = count(move for move in legal_moves if board.is_capture(move))
```
- **Range**: 0-15 typically
- **Tactical Indicator**: More captures = more forcing moves = higher complexity
- **Normalized**: scaled relative to 15 (max typical captures)

### 3.3 Checks Available (1 dim)
```python
checks = count(move for move in legal_moves if board.gives_check(move))
```
- **Range**: 0-5 typically
- **Attack Indicator**: Available checks = forcing attacks available
- **Tactical Nature**: Check moves force opponent response
- **Normalized**: scaled relative to 5

### 3.4 Piece Tension (1 dim)
```python
def calculate_piece_tension(board):
    """Count pieces under attack"""
    tension = 0
    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece and piece.piece_type != chess.KING:
            if board.is_attacked_by(not piece.color, square):
                tension += 1
    return tension  # 0-16 typically
```
- **Psychological Impact**: Many pieces under attack = unstable position
- **Normalized**: scaled relative to 16
- **Why**: Indicates sharp tactical play, uncertainty

### 3.5 Center Control (1 dim)
```python
def calculate_center_control(board):
    """Normalized difference in center square control (e4, d4, e5, d5)"""
    center_squares = [E4, D4, E5, D5]
    mover_control = count(squares_attacked_by_mover)
    opponent_control = count(squares_attacked_by_opponent)
    return (mover_control - opponent_control) / (mover_control + opponent_control)
    # Range: -1.0 (opponent owns center) to +1.0 (mover owns center)
```
- **Strategic**: Center control = space advantage = easier piece mobility
- **Principle**: Classic chess principle (control d4/e4 = positional plus)

### 3.6 Game Phase (1 dim)
```python
def calculate_game_phase(board):
    """0=opening, 0.5=midgame, 1=endgame (based on material)"""
    total_material = sum(piece_count * piece_value for all_pieces)
    starting_material = 60  # 2N+2B+2R+Q per side * 2
    phase = 1.0 - (total_material / starting_material)
    return min(1.0, max(0.0, phase))
```
- **Phases**:
  - Opening (0.0-0.2): Queens + rooks on board
  - Middlegame (0.2-0.7): Mixed piece endgames beginning
  - Endgame (0.7-1.0): Kings + minor pieces / pawns
- **Use**: Adjust evaluation weights by phase (king safety in opening, activity in endgame)

### 3.7 Move Diversity (1 dim)
```python
def calculate_move_diversity(board):
    """Count unique piece types that can move"""
    piece_types_moving = set()
    for move in board.legal_moves:
        piece = board.piece_at(move.from_square)
        piece_types_moving.add(piece.piece_type)
    return len(piece_types_moving)  # 1-6 range
```
- **Range**: 1-6 (can only move pawns vs can move all pieces)
- **Coordination**: More piece types active = more coordinated position
- **Flexibility**: Restricted piece types = harder to find good moves

### 3.8 Forest Darkness Score (1 dim) — **V7P3R Signature Metric**

```python
def calculate_forest_darkness(board):
    """V7P3R's custom complexity metric: "How dark is the forest?"
    Combines tactical density, piece tension, and move diversity.
    Higher = more complex/chaotic position (Tal would approve).
    """
    legal_moves = len(list(board.legal_moves))
    captures = sum(1 for m in board.legal_moves if board.is_capture(m))
    checks = sum(1 for m in board.legal_moves if board.gives_check(m))
    tension = calculate_piece_tension(board)
    diversity = calculate_move_diversity(board)
    
    # Normalize components
    legal_norm = min(legal_moves / 60.0, 1.0)      # Max ~60 moves
    capture_norm = min(captures / 15.0, 1.0)       # Max ~15 captures
    check_norm = min(checks / 5.0, 1.0)            # Max ~5 checks
    tension_norm = min(tension / 16.0, 1.0)        # Max ~16 pieces attacked
    diversity_norm = diversity / 6.0                # Max 6 piece types
    
    # Weighted combination (emphasize captures, checks, tension)
    darkness = (legal_norm * 0.2 + 
               capture_norm * 0.3 +    # Captures most important
               check_norm * 0.2 + 
               tension_norm * 0.2 +    # Tension equally important
               diversity_norm * 0.1)
    
    return darkness  # 0.0 (totally calm) to 1.0 (complete chaos)
```

**Philosophy**: "The darker the forest, the more room for tactical exploitation"
- V7P3R seeks complex positions where calculation edge matters more
- Higher values reward sharp, tactical positions
- Enables Tal-like play: embrace chaos where human judgment excels

---

## Part 4: Temporal Features (4 dims)

**Source**: v7.2 comprehensive features (added in v7 generational training)

These capture **time and situation context**—features that pure position analysis would miss.

### 4.1 Move Number (Normalized) (1 dim)
```python
move_number_normalized = min(move_number / 100.0, 1.0)
# Range: 0-1, where 1.0 = 100+ move game (endgame territory)
```
- **Why**: Endgame comes later in the game; early moves favor opening theory
- **Use**: Adjust opening book reliance, endgame complexity tolerance

### 4.2 Halfmove Clock (Normalized) (1 dim)
```python
halfmove_clock_normalized = (halfmove_clock / 50.0)
# Range: 0.0-1.0, where 0.98 = 49 halfmoves (about to trigger 50-move rule)
```
- **Context**: Proximity to 50-move rule (automatic draw)
- **Strategic**: Changes evaluation near 50-move rule
- **Urgency**: If at 45 halfmoves with advantage, must convert before draw

### 4.3 Urgency Score (1 dim)
```python
def calculate_urgency(board, position_complexity, is_advantage):
    """Simple position = high urgency to move fast (don't overthink)"""
    complexity = calculate_forest_darkness(board)
    legal_moves = len(list(board.legal_moves))
    
    if complexity < 0.2:  # Simple position
        urgency = 1.0  # Move fast
    elif complexity > 0.7:  # Chaotic position
        urgency = 0.1  # Think carefully
    else:
        urgency = 0.5
    
    return urgency  # 0.1 (think hard) to 1.0 (move fast)
```
- **Psychology**: Simple positions are "automatic" (less thinking needed)
- **Complexity = Uncertainty**: Chaotic positions need deep analysis
- **Use**: Time management in tournament play

### 4.4 Previous Inference Time (1 dim)
```python
previous_inference_time = last_move_think_time_seconds
# Range: 0.0 (opening first move) to 10.0+ (deep endgame analysis)
```
- **Feedback Loop**: Last move took 3 seconds? This move might too.
- **Consistency**: Monitor if engine is spending time appropriately
- **Use**: Training signal for move urgency calibration

---

## Part 5: Static Evaluator Heuristics (V7P3R Engines v11-v20)

**Source**: v7p3r-chess-engine lichess bots

These are the **deep heuristic vision** of the static evaluator—hand-tuned evaluation bonuses that capture chess principles.

### 5.1 King Safety Evaluator (from v12.4)

**Philosophy**: "King is your most important piece. Protect it early, activate it late."

#### Pawn Shelter Evaluation
```python
def evaluate_pawn_shelter(board, king_square, color):
    """Evaluate pawn shield strength (bonuses for pawns near king)"""
    pawn_shelter_bonus = [0, 5, 10, 15, 20]  # Index = num shelter pawns
    
    king_file = square_file(king_square)
    king_rank = square_rank(king_square)
    shelter_pawns = 0
    
    for file_offset in [-1, 0, 1]:
        check_file = king_file + file_offset
        if 0 <= check_file <= 7:
            # Look for pawn in front of king providing shelter
            shelter_found = False
            for rank in range(king_rank + 1, 8):  # White king case
                if pawn_at(square(check_file, rank)) == FRIENDLY_PAWN:
                    shelter_found = True
                    if rank - king_rank <= 2:  # Close pawns
                        shelter_pawns += 1
                    break
            
            if not shelter_found:
                return_score -= 10  # Penalty for missing shelter
    
    # Bonus for shield pawns
    return pawn_shelter_bonus[min(shelter_pawns, 4)]
```

**Values**:
- 0 shelter pawns: +0 cp
- 1 shelter pawn: +5 cp
- 2 shelter pawns: +10 cp
- 3 shelter pawns: +15 cp
- 4+ shelter pawns: +20 cp
- Missing shelter file: -10 cp

#### Castling Rights
```python
castling_rights_bonus = 25  # cp bonus for having castling available
```
- **Why**: Having options = flexibility = better position
- **Especially**: Early game when king safety is paramount

#### King Exposure
```python
king_exposure_penalty = 30  # cp penalty for exposed king
# Exposed = few escape squares, many enemy attackers nearby
```
- **Calculation**: Count escape squares, enemy pieces in attack zone
- **Scale**: Up to 30 cp if severely exposed

#### Escape Squares
```python
escape_square_bonus = 8  # cp per available escape square
# Escape square = square king can move to safely
```
- **Range**: 0-8 escape squares typically
- **Importance**: More escape squares = harder to checkmate

#### Attack Zone Penalty
```python
attack_zone_penalty = 12  # cp penalty per enemy piece attacking near king
# Attack zone = 3 squares around king
```
- **Cumulative**: Multiple attackers stack the penalty

#### Enemy Pawn Storms
```python
enemy_pawn_storm_penalty = 15  # cp for enemy pawn storm
advanced_enemy_pawn_penalty = 10  # cp per advanced enemy pawn
```
- **Pawn Storm**: Multiple enemy pawns advancing toward king
- **Advanced**: Pawns on 4th-5th rank (getting dangerous)

#### Endgame King Activity
```python
king_activity_bonus = 5  # cp bonus per centralization in endgame
king_centralization_bonus = [0, 2, 4, 8, 12, 8, 4, 2]  # By rank distance from center
```
- **When Active**: Endgames (material < 2000 cp)
- **Centralization**: King near center (d4-e4 area) = stronger
- **Mid-to-Corner**: King distance from center d4/e4 determines bonus

### 5.2 Advanced Pawn Structure Evaluator (from v12.4)

**Philosophy**: "Pawns are the soul of chess. Structure determines piece placement."

#### Passed Pawn Evaluation
```python
passed_pawn_bonus_by_rank = [0, 20, 30, 50, 80, 120, 180, 250]
# Index = rank (0-7 for white, flipped for black)

advanced_passed_bonus = 30  # Extra bonus if 6th rank or higher
connected_passed_bonus = 20  # Two passed pawns supporting each other
```

**Examples**:
- Passed pawn on 2nd rank: +20 cp
- Passed pawn on 5th rank: +80 cp
- Passed pawn on 7th rank (about to promote): +250 cp!
- Two connected passed pawns: +100+ cp total

#### Isolated Pawn Penalty
```python
isolated_pawn_penalty = 15  # cp per isolated pawn
pawn_on_open_file_extra_penalty = 10  # Isolated pawns on open files worse
```
- **Isolated**: No friendly pawns on adjacent files
- **Why Bad**: Cannot be defended by pawns; target for minor pieces

#### Doubled Pawn Penalty
```python
doubled_pawn_penalty = 25  # cp per doubled pawn
# Multiplied by (count - 1) for multiple pawns on same file
```
- **Examples**:
  - 2 pawns on e-file: -25 cp
  - 3 pawns on e-file: -50 cp
- **Why Bad**: Wasted pawn; only front pawn useful

#### Backward Pawn Penalty
```python
backward_pawn_penalty = 12  # cp per backward pawn
backward_on_semi_open_file_extra = 8  # Worse if opponent can attack
```
- **Backward**: Cannot advance safely (opponent controls advance square)
- **Weakness**: Sits in place, target for attack

#### Connected Pawn Bonus
```python
connected_pawn_bonus = 8  # cp per connected pawn (pawns protecting each other)
```
- **Why**: Can defend each other; harder to attack
- **Strong**: Connected pawns advance together

#### Pawn Chain Bonus
```python
pawn_chain_bonus = 5  # cp per pawn in chain (diagonal support)
# Chain = connected pawns in diagonal formation (like French setup)
```
- **Example**: a2-b3-c4 diagonal chain
- **Strength**: Base pawn can't be attacked (protected by whole chain)

#### Pawn Storms
```python
pawn_storm_bonus = 10  # cp for organized pawn attack
```
- **Storm**: Multiple pawns advancing toward enemy king
- **Tactical**: Creates weaknesses in enemy position

#### Pawn Shelter
```python
pawn_shelter_bonus = 15  # cp for friendly pawn shelter (redundant with king safety)
```
- **Same As**: King's pawn shield (counted in both)
- **Reinforces**: Value of castling into safety

### 5.3 Personality Reward System (from v7.0-v8.0)

**Philosophy**: "Stockfish is objectively best, but V7P3R has personality. Reward it for seeking complexity."

#### Configuration (v7.0 PersonalityWeights)
```python
@dataclass
class PersonalityWeights:
    # Complexity rewards
    forest_darkness: float = 0.15      # Reward high chaos scores
    piece_tension: float = 0.10        # Reward positions with tension
    move_diversity: float = 0.05       # Reward many piece types active
    
    # Material sacrifice tolerance
    material_sacrifice_bonus: float = 0.10
    material_threshold: int = 5        # Tolerate up to 5 pawn material loss
    complexity_threshold: float = 0.2  # If complexity gains 0.2+, reward sacrifice
    
    # King safety vs aggression
    king_risk_penalty: float = -0.05   # Penalty for king exposure
    king_risk_tolerance: float = 2.0   # Tolerate up to 2 pawn shield loss
    attack_bonus: float = 0.08         # Bonus if attacking enemy king
    
    # Strategic emphasis
    center_control: float = 0.05       # Standard chess
    passed_pawns: float = 0.03         # Standard chess
    bishop_pair: float = 0.02          # Standard chess
    active_rooks: float = 0.04         # Reward rook activity
    
    # Endgame adjustments
    endgame_complexity_weight: float = 0.5  # Reduce complexity seeking late
```

#### Reward Calculation Examples

**Example 1: Sacrifice for Complexity (Tal-Style)**
```
Position: White can sacrifice queen for 2 minor pieces + open lines
- Material loss: -3 cp (queen worth 9, two minors worth 6)
- Complexity gain: +0.3 forest darkness (more open, tactical)
- Reward: If complexity_gain (0.3) > threshold (0.2), reward sacrifice
- Result: Position gets bonus +0.10 * eval_scale
```

**Example 2: Attack King Despite Own King Exposure**
```
Position: Can attack black king, but own king becomes exposed
- King safety loss: -2 pawn shield (losing shelter pawns)
- Attack reward: +0.08 * eval_scale (attacking enemy king)
- King tolerance: 2.0 (can tolerate up to 2 pawn shield loss)
- Result: If shield loss < tolerance, reward aggressive attack
```

**Example 3: Piece Tension in Middlegame**
```
Position: Many pieces under attack (high piece tension)
- Piece tension score: 8 (8 pieces attacked)
- Reward: +0.10 * 8 * eval_scale = +0.80 bonus
- Why: V7P3R seeks sharp, tactical positions where calculation wins
```

---

## Summary: Heuristic Coverage

| Category | Dims | Source | Purpose |
|----------|------|--------|---------|
| Stage 1 Fast | 19 | v6.0+ | Piece-centric core |
| Heuristic | 24 | v7.0 | Positional quality |
| Complexity | 8 | v7.0 | Game difficulty |
| Temporal | 4 | v7.2 | Time context |
| **Total NN Input** | **55** | v7.0 | Complete position view |
| Static Engine | ∞ | v11-v20 | Hand-tuned bonuses |
| Personality | ∞ | v7.0-v8.0 | Tal-style rewards |

---

## Next Steps: Integration into v10.0

With this complete heuristic catalog, the v10.0 development plan will:

1. **HalfKA Board Features** (core): Board topology, piece positions
2. **Sentiment Vector** (appendix): Select custom heuristics (forest_darkness, piece_tension, king_safety, pawn_structure)
3. **Policy Network** (secondary): Move preference based on personality traits

This ensures V7P3R v10.0 inherits decades of heuristic wisdom while enabling neural networks to discover new patterns.

## Integration into v10.0: Unified Single Network Architecture

With the architectural decision finalized (Option B: Single Network with Dual Input Streams), the v10.0 implementation will follow this data and training pipeline:

### Data Serialization (Day 1: 3-4 days)
Both HalfKA and Sentiment extracted in parallel during raw data ingestion:

```
Game PGN
  │
  ├─→ HalfKA Extractor → 45K sparse indices per position
  │
  ├─→ Sentiment Extractor → 8-12 dense features per position
  │   ├─ forest_darkness from legal_moves, captures, checks, tension
  │   ├─ piece_tension from attacked pieces
  │   ├─ king_safety_score from king safety evaluator
  │   ├─ pawn_structure_score from pawn evaluator
  │   ├─ center_control from central square control
  │   ├─ game_phase from material remaining
  │   └─ move_urgency from position complexity
  │
  ├─→ Clock Extractor → 4 temporal features
  │
  └─→ Label Extractors (parallel)
      ├─ Stockfish evals (for eval head training)
      ├─ V7P3R move labels (for policy head training)
      └─ Syzygy W/D/L (for endgame head training)

Result: 88-byte record = HalfKA(sparse) + Sentiment(dense) + Clock(dense) + Labels
```

### Network Training (Day 3-4: 2 weeks)
Single unified network with three loss terms:

```
Single Training Loop:
  ├─ Forward pass: HalfKA → Eval head (MSE on Stockfish evals)
  ├─ Forward pass: Sentiment → Policy head (CE on V7P3R moves)
  ├─ Forward pass: Shared hidden → WDL head (CE on Syzygy truth)
  │
  ├─ Combined Loss = 0.7×eval_loss + 0.2×policy_loss + 0.1×wdl_loss
  │
  └─ Backprop through SHARED hidden layers
     ↓ Network learns to represent positions that satisfy all three objectives
```

### Inference: Time-Aware Blending (Integral to Model)

```python
def select_move_v10_0(board, halfdka, sentiment, clock_time):
    """Unified network, dynamic blending at inference."""
    
    # Single forward pass through network
    eval_score = model.eval_head(halfdka)      # cp score (objective)
    move_probs = model.policy_head(sentiment)  # move distribution (V7P3R style)
    wdl_probs = model.wdl_head(shared_hidden)  # endgame truth
    
    # Time-aware blending: sentiment can override eval
    time_ratio = clock_time / initial_clock
    complexity = sentiment['forest_darkness']
    
    if time_ratio < 0.1:          # <10% time
        sentiment_weight = 0.7
    elif time_ratio < 0.25:       # <25% time
        sentiment_weight = 0.5
    elif complexity > 0.6:        # complex position
        sentiment_weight = 0.4
    else:                         # plenty of time, simple position
        sentiment_weight = 0.2
    
    # Blend the signals
    final_probs = (1 - sentiment_weight) * eval_probs + \
                  sentiment_weight * policy_probs
    
    return select_best_move(final_probs)
```

### Key Architectural Properties

1. **No Separate Networks**: Single training pipeline, single forward pass at inference
2. **Gradient Interaction**: All three losses backprop through shared layers → learned representation balances all three objectives
3. **Sentiment Override**: Policy head (trained on V7P3R's moves) naturally learns to diverge from eval head in complex positions, and clock blending amplifies this divergence under time pressure
4. **Personality Preservation**: Policy head reproduces V7P3R's Tal-like style because it's trained on V7P3R's historical move choices, not on abstract rewards
5. **Evaluation Purity**: Eval head never sees clock data → evaluation remains objective for use in search algorithms

### Next Implementation Steps

✅ **Complete**: Heuristic catalog (55 neural features + static bonuses)
✅ **Complete**: Architectural decision (Option B: Unified single network)
✅ **Next**: Build HalfKA feature extractor (Day 2.1, ~100 lines)
✅ **Next**: Implement sentiment feature extraction (Day 2.2, ~150 lines)
✅ **Next**: Design accumulator architecture with sentiment encoding (Day 2.3, ~200 lines)
✅ **Next**: Write training loop with three-loss objective (Day 3.1-3.2, ~300 lines)
✅ **Next**: Build time-aware inference blender (Day 3.3, ~50 lines)
✅ **Next**: Quantization & deployment optimization (Day 4, ~100 lines)

## Visual: Unified Neural Network Architecture (Option B)

```
┌─────────────────────────────────────────────────────────────────┐
│         V10.0 Single Network with Dual Input Streams             │
└─────────────────────────────────────────────────────────────────┘

                    Board State (FEN)
                          │
           ┌──────────────┬─────────────┬──────────────┐
           │              │             │              │
           ▼              ▼             ▼              ▼
   HalfKA Features  Sentiment Features Clock Data  Move Context
   (45K sparse)    (8-12 dense)       (2-3 dims)    (history)
           │              │             │              │
           ▼              ▼             ▼              ▼
   ┌────────────┐ ┌──────────────┐ ┌─────────┐ ┌──────────┐
   │Accumulators│ │Dense Encoder │ │TimeEmbed│ │ History  │
   │(1024-2048) │ │(512-1024)    │ │ (256)   │ │ Encoder  │
   │ClippedReLU │ │ReLU, LayerNm │ │ ReLU    │ │ (256)    │
   └─────┬──────┘ └──────┬───────┘ └────┬────┘ └────┬─────┘
         │               │              │           │
         └───────────────┼──────────────┼───────────┘
                         │
          ┌──────────────▼──────────────┐
          │  Shared Hidden Layers       │  ◄─ Critical: BOTH inputs
          │  (2048-4096 neurons)        │      interact here through
          │  3-4 Residual Blocks        │      shared representation
          │  LayerNorm after blocks     │
          └──────────────┬──────────────┘
                         │
        ┌────────────────┼────────────────┐
        │                │                │
        ▼                ▼                ▼
   ┌────────────┐  ┌──────────────┐  ┌──────────┐
   │ Evaluation │  │ Character/   │  │   WDL    │
   │ Head       │  │ Policy Head  │  │   Head   │
   │ (64 out)   │  │ (256 out)    │  │ (32 out) │
   │ MSE Loss   │  │ CE Loss      │  │ CE Loss  │
   │   70%      │  │   20%        │  │   10%    │
   └─────┬──────┘  └──────┬───────┘  └────┬─────┘
         │                │               │
         ▼                ▼               ▼
    Eval Score      Move Probs      WDL Probs
    (cp)            (4000 moves)    (W/D/L)
         │                │               │
         └────────────────┼───────────────┘
                          │
         ┌────────────────▼────────────────┐
         │  Time-Aware Move Selection      │  ◄─ Inference Logic
         │                                 │
         │ If clock_time < 25%:            │
         │   sentiment_weight = 0.7        │
         │   Use Policy head more          │
         │                                 │
         │ If position complex (FD > 0.6): │
         │   sentiment_weight += 0.1       │
         │                                 │
         │ Blended Score =                 │
         │   (1-w)×eval_score +            │
         │   w×policy_score                │
         │                                 │
         │ Sentiment can override eval     │
         │ when time pressure detected     │
         └────────────────┬────────────────┘
                          │
                          ▼
                   Best Move Selected
```

---

## Why This Architecture (Option B: Unified Single Network)

**The key insight**: Sentiment features don't replace evaluation—they provide context for **when to trust intuition over calculation under time pressure**.

### Benefits of Single Network Architecture

| Aspect | Benefit |
|--------|---------|
| **Gradient Flow** | All three losses backpropagate through same hidden layers → shared representation learns features that benefit all three |
| **Feature Interaction** | HalfKA and sentiment interact in hidden layers → model discovers relationships between board topology and psychological state |
| **Inference Speed** | One forward pass instead of two separate networks |
| **Shared Understanding** | Network learns that complex positions (high forest_darkness) predict policy divergence from eval |
| **Time Awareness** | At inference, clock data modulates the blend of eval vs. policy → sentiment override happens naturally |
| **No Separate Training** | Single training loop with three losses → no need for separate pipelines |

### Training Process

```python
for board, move_label, eval_label, wdl_label in training_data:
    halfdka = extract_halfdka(board)
    sentiment = extract_sentiment(board)      # forest_darkness, tension, etc.
    clock_context = extract_clock(game_state)  # time remaining, opponent time
    
    # Forward pass through single network
    eval_out = model.eval_head(halfdka)
    policy_out = model.policy_head(sentiment)
    wdl_out = model.wdl_head(shared_hidden)
    
    # Three independent losses
    eval_loss = MSE(eval_out, eval_label)              # Stockfish evals
    policy_loss = CrossEntropy(policy_out, move_label)  # V7P3R's moves
    wdl_loss = CrossEntropy(wdl_out, wdl_label)        # Syzygy truth
    
    # Combined loss → ALL gradients flow through shared layers
    total_loss = 0.70 * eval_loss + 0.20 * policy_loss + 0.10 * wdl_loss
    total_loss.backward()  # ◄─ Updates shared representation
    optimizer.step()
```

The **shared hidden layers learn to represent positions in a way that serves all three objectives simultaneously**. Under gradient pressure, the network discovers that:

- High `forest_darkness` + `piece_tension` correlate with policy moves diverging from eval moves
- Time pressure amplifies the importance of `move_urgency` for policy selection
- Endgame (high `game_phase`) positions use different parts of the representation for W/D/L

---

## Feature Dimensions: All Inputs to Single Network

```
Input Channels to Unified V10.0 Model:

┌─ HALFDKA STREAM (Sparse) ───────────────────┐
│ • King bucket assignment (per perspective)   │
│ • Piece positions (one-hot in buckets)       │
│ • Perspective flip (white/black to move)     │
│ Result: ~45,000 sparse indices               │
├─ SENTIMENT STREAM (Dense) ─────────────────┤
│ • forest_darkness (0.0-1.0)                  │
│ • piece_tension (0-16)                       │
│ • king_safety_score (from evaluator)         │
│ • pawn_structure_score (passed/doubled/etc)  │
│ • center_control (-1.0 to +1.0)              │
│ • game_phase (0.0-1.0)                       │
│ • move_urgency (0.1-1.0)                     │
│ Result: 8-12 dense dimensions                │
├─ TEMPORAL STREAM (Dense) ──────────────────┤
│ • clock_remaining (seconds, normalized)      │
│ • opponent_clock_remaining (seconds)         │
│ • time_control_type (bullet/blitz/rapid)     │
│ • move_number (normalized)                   │
│ Result: 4 dense dimensions                   │
└─ MOVE HISTORY (Dense) ────────────────────┤
│ • Last 3 moves (piece→square embeddings)     │
│ • Move times (how long opponent thought)     │
│ Result: ~8 dense dimensions (optional)       │
└────────────────────────────────────────────┘

All streams merge in shared hidden layers where they interact.
```

---

## Integration: HalfKA + Sentiment in Data Pipeline

**Day 1 Data Serialization** (parallel extraction):
- Extract HalfKA: ✅ 45K sparse indices
- Extract Sentiment: ✅ 8-12 dense features
- Extract Clock: ✅ 4 temporal features
- Extract Labels: ✅ Stockfish evals, V7P3R moves, Syzygy W/D/L

**Day 3 Training**:
- Both HalfKA and Sentiment fed to model simultaneously
- Shared hidden layers learn joint representation
- Three losses optimize together

**Inference**:
- HalfKA → Eval head (objective position score)
- Sentiment → Policy head (human-like moves)
- Clock → Dynamic blending (how much to trust policy vs. eval)

---

## Key Architectural Decision: Why Not Separate Networks?

| Approach | Pros | Cons |
|----------|------|------|
| **Two Separate Networks** | Independent training | No interaction; slower inference; redundant computation |
| **Single Network, Dual Input** ✅ | Shared representation; gradient interaction; fast inference; elegant blending | More complex training pipeline |
| **Pure HalfKA Only** | Simplest; proven in alphazero | Loses all psychological context; can't override eval under time pressure |

**Decision: Single Network (Option B)** because the whole purpose of V10.0 is to let sentiment override evaluation under time pressure. That requires both input streams to influence move selection in a coordinated way. Separate networks can't achieve that naturally.